In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HOQ_E.xpt"
csv_file = DATA_DIR / "HOQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched released XPT for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 HOQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HOQ040",
    "HOD050",
    "HOD060",
    "HOQ065",
    "HOQ070",
    "HOQ080"
]


if df.shape != (10149, 7):
    raise ValueError(
        f"Unexpected HOQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "HOQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: HOQ_E structure = "
    "10,149 rows x 7 columns."
)


# ============================================================
# 4. FIELD CLASSIFICATION
# ============================================================

#
# Actual attached XPT:
#
# Genuine decimal variables = 0
#
# All variables are:
#
# - identifier
# - questionnaire categories
# - count/category fields
#
# Therefore all can be cleaned to Int64 after verifying
# every nonmissing source value is whole-number.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()

DECIMAL_COLUMNS = []


print(
    "\nInteger/code/count fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY-VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifact found. "
        "Do NOT automatically replace with zero."
    )


print(
    "PASS: HOQ_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 6. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "Unexpected true numeric zero found."
    )


print(
    "PASS: Original HOQ_E contains no numeric zeros."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:
    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:
    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):
    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 8. FRACTIONAL / DECIMAL CHECK
# ============================================================

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:
        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:
    raise ValueError(
        "STOP: Unexpected genuine decimal values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 9. CDC CHECK - HOQ040
# ============================================================

#
# When was home built
#
# CDC:
#
# 1  1990 to present     = 2,309
# 2  1978 to 1989        = 1,435
# 3  1960 to 1977        = 1,639
# 4  1950 to 1959        =   916
# 5  1940 to 1949        =   436
# 6  Before 1940         = 1,220
# 77 Refused             =     6
# 99 Don't know          = 2,094
# Missing                =    94
#

EXPECTED_HOQ040 = {
    1: 2309,
    2: 1435,
    3: 1639,
    4: 916,
    5: 436,
    6: 1220,
    77: 6,
    99: 2094
}


actual = (
    df["HOQ040"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_HOQ040:

    raise ValueError(
        "HOQ040 frequency mismatch.\n"
        f"Expected: {EXPECTED_HOQ040}\n"
        f"Actual:   {actual}"
    )


if int(
    df["HOQ040"]
    .isna()
    .sum()
) != 94:

    raise ValueError(
        "HOQ040 missing count differs."
    )


print(
    "\nPASS: HOQ040 matches CDC."
)


# ============================================================
# 10. CDC CHECK - HOD050
# ============================================================

#
# Number of rooms in home.
#
# CDC processing:
#
# Values >=13 were grouped and released as 13.
#
# CDC:
#
# 1-12 = 9,977
# 13   = 62
# 777  = 4
# 999  = 12
# Missing = 94
#

s = df["HOD050"]


range_count = int(
    s.between(
        1,
        12
    )
    .sum()
)


if range_count != 9977:

    raise ValueError(
        f"HOD050 1-12 count differs: "
        f"{range_count:,}"
    )


if int(
    s.eq(13)
    .sum()
) != 62:

    raise ValueError(
        "HOD050=13 count differs."
    )


if int(
    s.eq(777)
    .sum()
) != 4:

    raise ValueError(
        "HOD050=777 count differs."
    )


if int(
    s.eq(999)
    .sum()
) != 12:

    raise ValueError(
        "HOD050=999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 94:

    raise ValueError(
        "HOD050 missing count differs."
    )


print(
    "\nPASS: HOD050 matches CDC."
)

print(
    "      1-12 = 9,977"
)

print(
    "      13+ released as 13 = 62"
)

print(
    "      777 = 4"
)

print(
    "      999 = 12"
)

print(
    "      Missing = 94"
)


# ============================================================
# 11. CDC CHECK - HOD060
# ============================================================

#
# How many years family lived in home
#
# CDC states this was collected continuously but
# collapsed into categories for public release.
#
# Keep released category values AS IS.
#

EXPECTED_HOD060 = {
    1: 1650,
    2: 1973,
    3: 2010,
    4: 1641,
    5: 2770,
    77: 5,
    99: 6
}


actual = (
    df["HOD060"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_HOD060:

    raise ValueError(
        "HOD060 frequency mismatch.\n"
        f"Expected: {EXPECTED_HOD060}\n"
        f"Actual:   {actual}"
    )


if int(
    df["HOD060"]
    .isna()
    .sum()
) != 94:

    raise ValueError(
        "HOD060 missing count differs."
    )


print(
    "PASS: HOD060 matches CDC."
)


# ============================================================
# 12. CDC CHECK - HOQ065
# ============================================================

EXPECTED_HOQ065 = {
    1: 6083,
    2: 3820,
    3: 142,
    7: 3,
    9: 7
}


actual = (
    df["HOQ065"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_HOQ065:

    raise ValueError(
        "HOQ065 frequency mismatch."
    )


if int(
    df["HOQ065"]
    .isna()
    .sum()
) != 94:

    raise ValueError(
        "HOQ065 missing count differs."
    )


print(
    "PASS: HOQ065 matches CDC."
)


# ============================================================
# 13. CDC CHECK - HOQ070
# ============================================================

EXPECTED_HOQ070 = {
    1: 8891,
    2: 957,
    3: 54,
    7: 2,
    9: 151
}


actual = (
    df["HOQ070"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_HOQ070:

    raise ValueError(
        "HOQ070 frequency mismatch."
    )


if int(
    df["HOQ070"]
    .isna()
    .sum()
) != 94:

    raise ValueError(
        "HOQ070 missing count differs."
    )


print(
    "PASS: HOQ070 matches CDC."
)


# ============================================================
# 14. CDC CHECK - HOQ080
# ============================================================

EXPECTED_HOQ080 = {
    1: 2187,
    2: 7842,
    9: 26
}


actual = (
    df["HOQ080"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_HOQ080:

    raise ValueError(
        "HOQ080 frequency mismatch.\n"
        f"Expected: {EXPECTED_HOQ080}\n"
        f"Actual:   {actual}"
    )


if int(
    df["HOQ080"]
    .isna()
    .sum()
) != 94:

    raise ValueError(
        "HOQ080 missing count differs."
    )


if int(
    df["HOQ080"]
    .eq(7)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected HOQ080=7 value."
    )


print(
    "PASS: HOQ080 matches CDC."
)


# ============================================================
# 15. IMPORTANT RELEASE-PROCESSING NOTES
# ============================================================

#
# CDC already performed these public-release transformations:
#
# HOD050:
# responses >=13 grouped into 13
#
# HOD060:
# continuous responses collapsed into categories
#
# HOQ080:
# detailed treatment-device combinations not released
#
#
# DO NOT:
#
# - reconstruct original HOD050 values >13
# - turn HOD060 back into continuous values
# - infer specific HOQ080 devices
#
# Released XPT is source of truth.
#

print(
    "\nPASS: Public-release processing retained AS IS."
)


# ============================================================
# 16. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# HOQ_E requires NO source-value correction.
#
# Tiny artifacts = 0
# Zero restorations = 0
# Genuine decimals = 0
#
# Therefore expected_df equals the original XPT.
#

expected_df = source_df.copy()


# ============================================================
# 17. VERIFY ALL INTEGER FIELDS ARE WHOLE
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):

        raise ValueError(
            f"STOP: {col} contains non-whole values."
        )


print(
    "\nPASS: All 7 released HOQ_E variables "
    "contain whole-number source values."
)


# ============================================================
# 18. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# Examples:
#
# SEQN:
# 41475.0 -> 41475
#
# HOQ040:
# 1.0 -> 1
#
# HOD050:
# 13.0 -> 13
#
# HOD060:
# 5.0 -> 5
#
# Missing:
# NaN -> blank
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All HOQ_E fields converted "
    "to clean integer-style formatting."
)


# ============================================================
# 19. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected pre-export changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal original XPT exactly."
)


# ============================================================
# 20. EXPORT CSV
# ============================================================

#
# No:
#
# round()
# float_format
# recoding
# filling missing values
#
# Missing source values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 21. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 22. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 23. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 7 columns numerically "
    "match original XPT exactly."
)


# ============================================================
# 24. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 25. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 10149:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 26. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# All HOQ_E fields are whole-number source values.
#
# expected raw representation:
#
# str(int(source_value))
#
# Examples:
#
# 41475.0 -> "41475"
# 1.0     -> "1"
# 13.0    -> "13"
# 999.0   -> "999"
#
# Missing -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:
        print(example)


    raise ValueError(
        "HOQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected source representation."
)


# ============================================================
# 27. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 28. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 29. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 30. FINAL CDC FREQUENCY RECHECK
# ============================================================

FINAL_EXPECTED = {
    "HOQ040": EXPECTED_HOQ040,
    "HOD060": EXPECTED_HOD060,
    "HOQ065": EXPECTED_HOQ065,
    "HOQ070": EXPECTED_HOQ070,
    "HOQ080": EXPECTED_HOQ080
}


for col, expected_counts in (
    FINAL_EXPECTED.items()
):

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"Final {col} frequencies changed."
        )


    if int(
        csv_df[col]
        .isna()
        .sum()
    ) != 94:

        raise ValueError(
            f"Final {col} missing count changed."
        )


# HOD050 separately because CDC publishes 1-12 as range

s = csv_df["HOD050"]


if not (
    int(
        s.between(
            1,
            12
        ).sum()
    ) == 9977

    and

    int(
        s.eq(13).sum()
    ) == 62

    and

    int(
        s.eq(777).sum()
    ) == 4

    and

    int(
        s.eq(999).sum()
    ) == 12

    and

    int(
        s.isna().sum()
    ) == 94
):

    raise ValueError(
        "Final HOD050 distribution changed."
    )


print(
    "\nPASS: Final HOQ_E frequencies "
    "match original XPT/CDC."
)


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HOQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)

print(
    "Genuine decimal variables:",
    0
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "HOQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 HOQ_E XPT -> CSV
Rows: 10,149
Columns: 7
PASS: HOQ_E structure = 10,149 rows x 7 columns.

Integer/code/count fields: ['SEQN', 'HOQ040', 'HOD050', 'HOD060', 'HOQ065', 'HOQ070', 'HOQ080']
Genuine decimal fields: []

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: HOQ_E contains NO tiny XPORT artifacts.

True numeric zeros: 0
PASS: Original HOQ_E contains no numeric zeros.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

PASS: HOQ040 matches CDC.

PASS: HOD050 matches CDC.
      1-12 = 9,977
      13+ released as 13 = 62
      777 = 4
      999 = 12
      Missing = 94
PASS: HOD060 matches CDC.
PASS: HOQ065 matches CDC.
PASS: HOQ070 matches CDC.
PASS: HOQ080 matches CDC.

PASS: Public-release processing retained AS IS.

PASS: All 7 released HOQ_E variables contain whole-number source values.
PASS: All HOQ_E field